# Exploración del Dataset Olist
**Pipeline de Datos en Tiempo Real para Análisis de Riesgo de Impago en E-Commerce**

Universidad Privada del Valle (UNIVALLE) · Ingeniería de Datos · 2026  
Juan Felipe Caballero Flores · Luciana Sofía Coca Terrazas  
Docente: M.Sc. Ing. Oscar Contreras Carrasco

---
Este notebook explora las 9 tablas `raw_*` cargadas en PostgreSQL desde los CSVs de Olist.  
El análisis justifica las decisiones de limpieza aplicadas en la capa Silver (dbt).

## Celda 1 · Imports y conexión a PostgreSQL

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Conexión ──────────────────────────────────────────────────────────────────
DB_USER     = 'olist_user'
DB_PASSWORD = 'olist_pass'
DB_HOST     = 'localhost'
DB_PORT     = '5433'       # Puerto mapeado en docker-compose (5433 → 5432 interno)
DB_NAME     = 'olist_db'

engine = create_engine(
    f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    pool_pre_ping=True
)

with engine.connect() as conn:
    result = conn.execute(text('SELECT version()')).fetchone()
    print('✔ Conexión exitosa:', result[0][:60])

# ── Cargar todas las tablas raw_ ──────────────────────────────────────────────
TABLAS = [
    'raw_orders',
    'raw_customers',
    'raw_order_items',
    'raw_order_payments',
    'raw_order_reviews',
    'raw_products',
    'raw_sellers',
    'raw_geolocation',
    'raw_product_category_translation',
]

dfs = {}
for tabla in TABLAS:
    dfs[tabla] = pd.read_sql(f'SELECT * FROM {tabla}', engine)
    print(f'  {tabla}: {len(dfs[tabla]):>9,} filas')

print('\n✔ Todas las tablas cargadas en memoria.')

## Celda 2 · Forma de cada tabla: filas, columnas, tipos de datos

In [ ]:
print('=' * 65)
print(f'{'TABLA':<38} {'FILAS':>8}  {'COLS':>5}  DTYPES ÚNICOS')
print('=' * 65)
for nombre, df in dfs.items():
    dtypes_resumen = ', '.join(sorted(df.dtypes.astype(str).unique()))
    print(f'{nombre:<38} {len(df):>8,}  {len(df.columns):>5}  {dtypes_resumen}')
print('=' * 65)

In [ ]:
# .info() detallado para las tablas principales
for nombre in ['raw_orders', 'raw_order_payments', 'raw_order_reviews']:
    print(f'\n{'─' * 55}')
    print(f'  {nombre}')
    print('─' * 55)
    dfs[nombre].info(show_counts=True)

## Celda 3 · Conteo y porcentaje de nulos por tabla

In [ ]:
registros_nulos = []

for nombre, df in dfs.items():
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    for col, n in nulos.items():
        registros_nulos.append({
            'tabla':   nombre,
            'columna': col,
            'nulos':   n,
            'pct':     round(n / len(df) * 100, 2)
        })

df_nulos = pd.DataFrame(registros_nulos).sort_values('pct', ascending=False)

if df_nulos.empty:
    print('No se encontraron valores nulos.')
else:
    print(f'Columnas con valores nulos: {len(df_nulos)}\n')
    print(df_nulos.to_string(index=False))

In [ ]:
# Visualización: top 12 columnas con más nulos
top_nulos = df_nulos.head(12).copy()
top_nulos['etiqueta'] = top_nulos['tabla'].str.replace('raw_', '') + '.' + top_nulos['columna']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_nulos['etiqueta'], top_nulos['pct'], color='#E05C5C', edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)
ax.set_xlabel('% de valores nulos')
ax.set_title('Top columnas con valores nulos — Dataset Olist', fontweight='bold')
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.axvline(x=50, color='gray', linestyle='--', alpha=0.4, label='50%')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../docs/screenshots/fig_nulos.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada en docs/screenshots/fig_nulos.png')

## Celda 4 · Distribuciones clave

In [ ]:
# ── 4a. Histograma de payment_value ──────────────────────────────────────────
pagos = dfs['raw_order_payments']['payment_value'].dropna()
pagos_filtrados = pagos[pagos <= 1000]   # excluye outliers extremos para visualizar

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(pagos_filtrados, bins=60, color='#4A90D9', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribución payment_value (≤ 1000 BRL)', fontweight='bold')
axes[0].set_xlabel('Valor del pago (BRL)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(pagos.median(), color='orange', linestyle='--', label=f'Mediana: {pagos.median():.0f}')
axes[0].axvline(pagos.mean(),   color='red',    linestyle='--', label=f'Media:   {pagos.mean():.0f}')
axes[0].legend(fontsize=9)

axes[1].boxplot(pagos, vert=True, patch_artist=True,
               boxprops=dict(facecolor='#4A90D9', alpha=0.6))
axes[1].set_title('Boxplot payment_value (todos los valores)', fontweight='bold')
axes[1].set_ylabel('Valor del pago (BRL)')
axes[1].set_xticks([])

plt.suptitle('Análisis de payment_value — raw_order_payments', y=1.01, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/screenshots/fig_payment_value.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nEstadísticas de payment_value:')
print(pagos.describe().round(2))
print(f'\nPagos con value = 0:  {(pagos == 0).sum()} registros')
print(f'Outliers (> IQR*1.5): {(pagos > pagos.quantile(0.75) + 1.5*(pagos.quantile(0.75)-pagos.quantile(0.25))).sum()} registros')

In [ ]:
# ── 4b. Barras de order_status ────────────────────────────────────────────────
status_counts = dfs['raw_orders']['order_status'].value_counts()

COLORES_STATUS = {
    'delivered':          '#2ECC71',
    'shipped':            '#3498DB',
    'canceled':           '#E74C3C',
    'unavailable':        '#E74C3C',
    'processing':         '#F39C12',
    'invoiced':           '#9B59B6',
    'approved':           '#1ABC9C',
    'created':            '#BDC3C7',
}
colores = [COLORES_STATUS.get(s, '#95A5A6') for s in status_counts.index]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(status_counts.index, status_counts.values, color=colores, edgecolor='white')
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_title('Distribución de order_status — raw_orders', fontweight='bold')
ax.set_xlabel('Estado del pedido')
ax.set_ylabel('Cantidad de pedidos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('../docs/screenshots/fig_order_status.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nDistribución de order_status:')
resumen_status = status_counts.to_frame('cantidad')
resumen_status['porcentaje'] = (resumen_status['cantidad'] / resumen_status['cantidad'].sum() * 100).round(2)
print(resumen_status.to_string())

In [ ]:
# ── 4c. Barras de review_score ────────────────────────────────────────────────
score_counts = dfs['raw_order_reviews']['review_score'].value_counts().sort_index()

COLORES_SCORE = ['#E74C3C', '#E67E22', '#F1C40F', '#2ECC71', '#27AE60']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(score_counts.index.astype(str), score_counts.values,
              color=COLORES_SCORE, edgecolor='white', width=0.6)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=10)
ax.set_title('Distribución de review_score — raw_order_reviews', fontweight='bold')
ax.set_xlabel('Puntuación (1 = muy malo, 5 = excelente)')
ax.set_ylabel('Cantidad de reseñas')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('../docs/screenshots/fig_review_score.png', dpi=120, bbox_inches='tight')
plt.show()

pct_negativas = score_counts[score_counts.index <= 2].sum() / score_counts.sum() * 100
print(f'Reviews con score ≤ 2 (flag_review_negativa): {pct_negativas:.1f}%')
print(f'Nulos en review_comment_message: {dfs["raw_order_reviews"]["review_comment_message"].isnull().mean()*100:.1f}%')

## Celda 5 · Relaciones entre tablas: integridad referencial

In [ ]:
orders_ids    = set(dfs['raw_orders']['order_id'])
payments_ids  = set(dfs['raw_order_payments']['order_id'])
items_ids     = set(dfs['raw_order_items']['order_id'])
reviews_ids   = set(dfs['raw_order_reviews']['order_id'])
customers_ids = set(dfs['raw_customers']['customer_id'])
orders_cust   = set(dfs['raw_orders']['customer_id'])

print('=' * 60)
print('ANÁLISIS DE INTEGRIDAD REFERENCIAL')
print('=' * 60)

# orders → payments
sin_pago = orders_ids - payments_ids
print(f'\nOrders SIN registro en payments: {len(sin_pago):,}')
if sin_pago:
    sample = dfs['raw_orders'][dfs['raw_orders']['order_id'].isin(sin_pago)]
    print(f'  Distribución por status:\n{sample["order_status"].value_counts().to_string()}')

# orders → order_items
sin_item = orders_ids - items_ids
print(f'\nOrders SIN registro en order_items: {len(sin_item):,}')

# orders → order_reviews
sin_review = orders_ids - reviews_ids
print(f'\nOrders SIN review: {len(sin_review):,} ({len(sin_review)/len(orders_ids)*100:.1f}%)')

# customers → orders (orphan customers)
customers_sin_orden = customers_ids - orders_cust
print(f'\nCustomers SIN ningún order: {len(customers_sin_orden):,}')

# Pagos huérfanos (en payments pero no en orders)
pagos_huerfanos = payments_ids - orders_ids
print(f'\nPayments SIN order en raw_orders (huérfanos): {len(pagos_huerfanos):,}')

print('\n' + '=' * 60)
print('RESUMEN DE COBERTURA')
print('=' * 60)
print(f'  raw_orders       → {len(orders_ids):>7,} order_ids únicos')
print(f'  raw_order_payments → {len(payments_ids):>5,} order_ids únicos')
print(f'  raw_order_items  → {len(items_ids):>7,} order_ids únicos')
print(f'  raw_order_reviews→ {len(reviews_ids):>7,} order_ids únicos')

In [ ]:
# ── Análisis de duplicados ────────────────────────────────────────────────────
print('DUPLICADOS POR TABLA')
print('-' * 45)
for nombre, df in dfs.items():
    dupes = df.duplicated().sum()
    print(f'  {nombre:<38} {dupes:>6,} ({dupes/len(df)*100:.2f}%)')

In [ ]:
# ── Inconsistencias temporales en raw_orders ─────────────────────────────────
orders = dfs['raw_orders'].copy()

# Asegurar tipos datetime
for col in ['order_purchase_timestamp', 'order_delivered_customer_date',
            'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Pedidos entregados antes de la compra (imposible)
entrega_antes_compra = orders[
    orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']
]
print(f'Pedidos con entrega < compra (inconsistencia crítica): {len(entrega_antes_compra)}')

# Días de atraso
entregados = orders[orders['order_status'] == 'delivered'].copy()
entregados['dias_atraso'] = (
    entregados['order_delivered_customer_date'] - entregados['order_estimated_delivery_date']
).dt.days

n_atrasados    = (entregados['dias_atraso'] > 0).sum()
n_adelantados  = (entregados['dias_atraso'] <= 0).sum()
n_atraso_grave = (entregados['dias_atraso'] > 30).sum()

print(f'\nPedidos delivered con atraso real (días_atraso > 0):  {n_atrasados:,}')
print(f'Pedidos delivered entregados antes del estimado:       {n_adelantados:,}')
print(f'Pedidos con atraso grave (> 30 días):                  {n_atraso_grave:,}')

## Celda 6 · Resumen de hallazgos y decisiones de limpieza

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║          RESUMEN DE HALLAZGOS — DATASET OLIST                   ║
╠══════════════════════════════════════════════════════════════════╣

HALLAZGO 1 — Nulos críticos en review_comment
  • review_comment_message: 58.7% nulos
  • review_comment_title:   88.3% nulos
  Decisión Silver: no eliminar filas; imputar con cadena vacía ''
  y agregar flag has_comment (0/1). Los scores numéricos son válidos.

HALLAZGO 2 — Duplicados masivos en geolocation
  • 1,000,163 filas → 261,831 duplicados exactos (26.2%)
  • Causa: múltiples coordenadas GPS por ZIP code
  Decisión Silver: deduplicar por zip_code tomando el centroide
  (mean de lat/lng). Resultado: 19,015 ZIPs únicos.

HALLAZGO 3 — Pedidos sin registro de pago
  • ~775 order_ids en raw_orders sin fila en raw_order_payments
  • Todos tienen status != 'delivered' (canceled, processing, etc.)
  Decisión Silver: mantener en stg_orders con payment_value = NULL.
  En Gold, flag_pago_cero = 1 activa flag_riesgo.

HALLAZGO 4 — Outliers en payment_value y price
  • payment_value: 7,981 registros > 344 BRL (límite IQR)
  • 9 pagos con value = 0 en pedidos 'delivered' (anomalía)
  • price en order_items: 8,427 outliers > 277 BRL
  Decisión Silver: NO eliminar outliers (son válidos en e-commerce).
  Marcar los payment_value = 0 con flag_pago_cero = 1.

HALLAZGO 5 — Distribución sesgada de review_score
  • 57,328 reseñas con score = 5 (57.8% del total)
  • Solo ~12% de reseñas con score ≤ 2 (flag_review_negativa)
  Decisión Silver: calcular flag_review_negativa = (score <= 2).
  En Gold, este flag contribuye a flag_riesgo del pedido.

╠══════════════════════════════════════════════════════════════════╣
║  FLAG_RIESGO = 1  cuando cualquiera de estos es verdadero:      ║
║    • order_status IN ('canceled', 'unavailable')                 ║
║    • review_score <= 2                                           ║
║    • dias_atraso > 30                                            ║
║    • payment_value = 0 con status 'delivered'                    ║
╚══════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ── Estimación de pedidos que activarán flag_riesgo ──────────────────────────
orders_analisis = dfs['raw_orders'].copy()
reviews_analisis = dfs['raw_order_reviews'][['order_id', 'review_score']].copy()
pagos_analisis   = dfs['raw_order_payments'].groupby('order_id')['payment_value'].sum().reset_index()

merged = orders_analisis.merge(reviews_analisis, on='order_id', how='left')
merged = merged.merge(pagos_analisis, on='order_id', how='left')

# Calcular dias_atraso
for col in ['order_delivered_customer_date', 'order_estimated_delivery_date']:
    merged[col] = pd.to_datetime(merged[col], errors='coerce')
merged['dias_atraso'] = (merged['order_delivered_customer_date'] - merged['order_estimated_delivery_date']).dt.days

# flag_riesgo
merged['flag_riesgo'] = (
    (merged['order_status'].isin(['canceled', 'unavailable'])) |
    (merged['review_score'] <= 2) |
    (merged['dias_atraso'] > 30) |
    ((merged['payment_value'] == 0) & (merged['order_status'] == 'delivered'))
).astype(int)

total_riesgo = merged['flag_riesgo'].sum()
pct_riesgo   = total_riesgo / len(merged) * 100

print(f'Pedidos con flag_riesgo = 1: {total_riesgo:,} ({pct_riesgo:.1f}% del total)')
print(f'Pedidos con flag_riesgo = 0: {len(merged) - total_riesgo:,} ({100-pct_riesgo:.1f}%)')

# Desglose por causa
print('\nDesglose por causa de riesgo (un pedido puede tener varias):')
print(f'  - Status canceled/unavailable: {merged["order_status"].isin(["canceled","unavailable"]).sum():,}')
print(f'  - Review score <= 2:           {(merged["review_score"] <= 2).sum():,}')
print(f'  - Atraso > 30 días:            {(merged["dias_atraso"] > 30).sum():,}')
print(f'  - Pago = 0 en delivered:       {((merged["payment_value"]==0)&(merged["order_status"]=="delivered")).sum():,}')

In [ ]:
# Cerrar la conexión al finalizar
engine.dispose()
print('✔ Exploración completada. Conexión a PostgreSQL cerrada.')
print('\nPróximo paso → Semana 2: modelos dbt Silver y Gold')